# Structured credit (tranche + correlation intuition)

**Prerequisites:** **06**. Full **`structured_credit`** waterfall JSON evolves quickly; here we price a **`cds_tranche`** as a **correlation-sensitive tranche** proxy and discuss CLO/ABS concepts in narrative form.


## Concept

- **ABS/CLO/CMBS:** pool cashflows, tranches (attachment/detachment), waterfall priorities.
- **OC/IC tests:** divert cash when collateral quality or interest coverage weakens.

A **CDS tranche** on an index uses **base correlation** to price mezzanine/equity risk—useful for the same dependency modeling mindset.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, HazardCurve, MarketContext

print("Imports OK (structured credit / tranche).")


## Instrument JSON (`cds_tranche` proxy)

**Attachment / detachment:** In this JSON, `attach_pct` and `detach_pct` are **index notional fractions in percentage points** on the same scale as base-correlation `detachment_points` (e.g. `detach_pct: 3.0` means a **3%** detachment / 0–3% tranche; in decimal terms that width is **0.03**).

**Equity tranche coupon:** Under **ISDA** standardized tranche conventions, **equity** (first-loss) tranches often carry a **higher running coupon** than mezzanine slices; the demo below uses **500 bp** as a stylized equity coupon for illustration.


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

tranche = json.dumps(
    {
        "type": "cds_tranche",
        "spec": {
            "id": "CDX-0-3",
            "index_name": "CDX.NA.IG",
            "series": 42,
            "attach_pct": 0.0,
            "detach_pct": 3.0,
            "notional": {"amount": "10000000", "currency": "USD"},
            "maturity": "2029-12-20",
            "running_coupon_bp": 500.0,
            "frequency": {"count": 3, "unit": "months"},
            "day_count": "Act360",
            "bdc": "following",
            "calendar_id": "weekends_only",
            "discount_curve_id": "USD-OIS",
            "credit_index_id": "CDX.NA.IG.HAZARD",
            "side": "buy_protection",
            "accumulated_loss": 0.0,
            "standard_imm_dates": False,
            "attributes": {"tags": [], "meta": {}},
        },
    }
)

try:
    validate_instrument_json(tranche)
    print("cds_tranche JSON OK")
except Exception as exc:
    print("validate:", type(exc).__name__, exc)


## MarketContext (hazard + base correlation + credit index)


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
haz = HazardCurve(
    "CDX-HAZ",
    AS_OF,
    [(1.0, 0.01), (5.0, 0.02), (10.0, 0.025)],
    recovery_rate=0.4,
)
mc = MarketContext().insert(ois).insert(haz)
state = json.loads(mc.to_json())
state["curves"].append(
    {
        "type": "base_correlation",
        "id": "CDX-BC",
        "detachment_points": [3.0, 7.0, 10.0, 15.0, 30.0, 100.0],
        "correlations": [0.18, 0.28, 0.35, 0.45, 0.65, 0.85],
    }
)
state["credit_indices"] = [
    {
        "id": "CDX.NA.IG.HAZARD",
        "num_constituents": 125,
        "recovery_rate": 0.4,
        "index_credit_curve_id": "CDX-HAZ",
        "base_correlation_curve_id": "CDX-BC",
    }
]
market_json = json.dumps(state)
print("credit_indices:", len(state["credit_indices"]))


## Pricing


In [ ]:
try:
    out = price_instrument_with_metrics(tranche, market_json, AS_OF_STR, model="hazard_rate")
    print(ValuationResult.from_json(out))
except Exception as exc:
    print("price:", type(exc).__name__, exc)


## Metrics


In [ ]:
try:
    out = price_instrument_with_metrics(
        tranche,
        market_json,
        AS_OF_STR,
        model="hazard_rate",
        metrics=["correlation01", "expected_loss", "jump_to_default"],
    )
    vr = ValuationResult.from_json(out)
    for m in ("correlation01", "expected_loss", "jump_to_default"):
        v = vr.get_metric(m)
        if v is not None:
            print(m, ":", v)
except Exception as exc:
    print("metrics:", type(exc).__name__, exc)
print("CLO waterfall JSON: see finstack/valuations/schemas/instruments/1/fixed_income/structured_credit.schema.json")


## Takeaways

- **Tranche PV** depends on **hazard** and **base correlation** shapes.
- **Structured credit deals** add pool-specific waterfalls—validate against the latest schema before relying on production JSON.
- Use this **cds_tranche** exercise as a **correlation / subordination** teaching analog.
